# Abnormal Vendor Detection Notebook

This notebook detects abnormal vendor behavior and potential ghost vendors using invoice statistics, vendor age, bank-account duplication, and payment concentration metrics.

## 1. Load vendor and invoice data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
pd.options.display.float_format = "{:,.0f}".format

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "03_Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "03_Data"
OUTPUT_DIR = PROJECT_ROOT / "05_Python_Analysis" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def yen(value):
    return f"¥{value:,.0f}"

vendors = pd.read_csv(DATA_DIR / "vendors.csv", parse_dates=["registration_date"])
invoices = pd.read_csv(DATA_DIR / "invoices.csv", parse_dates=["invoice_date", "posting_date", "payment_date"])

vendor_invoices = invoices.merge(vendors, on="vendor_id", how="left")
analysis_date = max(vendor_invoices["invoice_date"].max(), vendors["registration_date"].max())
analysis_date

## 2. Calculate vendor behavior metrics

In [ ]:
vendor_metrics = (
    vendor_invoices.groupby(["vendor_id", "vendor_name"], as_index=False)
    .agg(
        invoice_count=("invoice_id", "count"),
        total_invoice_value_jpy=("invoice_amount", "sum"),
        average_invoice_amount_jpy=("invoice_amount", "mean"),
        median_invoice_amount_jpy=("invoice_amount", "median"),
        max_invoice_amount_jpy=("invoice_amount", "max"),
        first_invoice_date=("invoice_date", "min"),
        last_invoice_date=("invoice_date", "max"),
        paid_value_jpy=("invoice_amount", lambda s: s[vendor_invoices.loc[s.index, "payment_status"].eq("Paid")].sum()),
        price_variance_count=("three_way_match_status", lambda s: s.eq("Price Variance").sum()),
        no_po_count=("three_way_match_status", lambda s: s.eq("No PO").sum()),
    )
)

vendor_metrics = vendor_metrics.merge(
    vendors[["vendor_id", "registration_date", "vendor_category", "bank_account_number", "is_active", "is_ghost_vendor", "similar_to_vendor"]],
    on="vendor_id",
    how="left",
)
vendor_metrics["days_since_registration"] = (analysis_date - vendor_metrics["registration_date"]).dt.days
vendor_metrics["invoice_frequency_per_month"] = vendor_metrics["invoice_count"] / ((analysis_date - vendor_metrics["first_invoice_date"]).dt.days.clip(lower=1) / 30)
vendor_metrics.head()

In [ ]:
monthly_vendor_payments = (
    vendor_invoices.assign(month=vendor_invoices["invoice_date"].dt.to_period("M").dt.to_timestamp())
    .groupby(["vendor_id", "vendor_name", "month"], as_index=False)
    .agg(monthly_invoice_value_jpy=("invoice_amount", "sum"), monthly_invoice_count=("invoice_id", "count"))
)

payment_trend = monthly_vendor_payments.sort_values(["vendor_id", "month"]).copy()
payment_trend["prior_month_value_jpy"] = payment_trend.groupby("vendor_id")["monthly_invoice_value_jpy"].shift(1)
payment_trend["month_over_month_change_jpy"] = payment_trend["monthly_invoice_value_jpy"] - payment_trend["prior_month_value_jpy"]
payment_trend.head(15)

## 3. Statistical anomaly detection

In [ ]:
# Vendor-level Z-score: compares each vendor's average invoice size against the portfolio distribution.
portfolio_mean = vendor_metrics["average_invoice_amount_jpy"].mean()
portfolio_std = vendor_metrics["average_invoice_amount_jpy"].std(ddof=0)
vendor_metrics["avg_invoice_z_score"] = (vendor_metrics["average_invoice_amount_jpy"] - portfolio_mean) / portfolio_std
vendor_metrics["abnormal_amount_vendor"] = vendor_metrics["avg_invoice_z_score"].abs() > 2.5

invoice_stats = invoices.groupby("vendor_id")["invoice_amount"].agg(["mean", "std"]).rename(columns={"mean": "vendor_invoice_mean", "std": "vendor_invoice_std"})
invoice_level = vendor_invoices.merge(invoice_stats, on="vendor_id", how="left")
invoice_level["invoice_amount_z_score"] = (invoice_level["invoice_amount"] - invoice_level["vendor_invoice_mean"]) / invoice_level["vendor_invoice_std"].replace(0, pd.NA)
invoice_level["invoice_amount_z_score"] = invoice_level["invoice_amount_z_score"].fillna(0)
invoice_level["is_invoice_outlier"] = invoice_level["invoice_amount_z_score"].abs() > 2.5
invoice_level.to_csv(OUTPUT_DIR / "abnormal_invoice_outliers.csv", index=False)
invoice_level[invoice_level["is_invoice_outlier"]].head(20)

In [ ]:
amount_mean = vendor_metrics["total_invoice_value_jpy"].mean()
amount_std = vendor_metrics["total_invoice_value_jpy"].std(ddof=0)
vendor_metrics["new_vendor_high_payment"] = (
    (vendor_metrics["days_since_registration"] < 90)
    & (vendor_metrics["total_invoice_value_jpy"] > amount_mean + 2 * amount_std)
)

vendor_metrics["duplicate_bank_account"] = vendor_metrics.duplicated("bank_account_number", keep=False)
vendor_metrics["ghost_vendor_indicator"] = (
    vendor_metrics["new_vendor_high_payment"]
    | vendor_metrics["duplicate_bank_account"]
    | vendor_metrics["is_ghost_vendor"].astype(str).eq("True")
)
vendor_metrics.sort_values("avg_invoice_z_score", ascending=False).head(15)

## 4. Risk categorization

In [ ]:
vendor_metrics["amount_score"] = vendor_metrics["avg_invoice_z_score"].abs().rank(pct=True) * 25
vendor_metrics["volume_score"] = vendor_metrics["total_invoice_value_jpy"].rank(pct=True) * 25
vendor_metrics["frequency_score"] = vendor_metrics["invoice_frequency_per_month"].rank(pct=True) * 20
vendor_metrics["new_vendor_score"] = vendor_metrics["new_vendor_high_payment"].astype(int) * 15
vendor_metrics["bank_score"] = vendor_metrics["duplicate_bank_account"].astype(int) * 10
vendor_metrics["ghost_label_score"] = vendor_metrics["is_ghost_vendor"].astype(str).eq("True").astype(int) * 5
vendor_metrics["vendor_risk_score"] = (
    vendor_metrics[["amount_score", "volume_score", "frequency_score", "new_vendor_score", "bank_score", "ghost_label_score"]].sum(axis=1)
).round(1)
vendor_metrics["risk_category"] = pd.cut(
    vendor_metrics["vendor_risk_score"],
    bins=[0, 45, 70, 100],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)
vendor_scorecard = vendor_metrics.sort_values("vendor_risk_score", ascending=False)
vendor_scorecard.to_csv(OUTPUT_DIR / "abnormal_vendor_risk_scorecard.csv", index=False)
vendor_scorecard.head(20)

## 5. Ghost vendor identification

In [ ]:
ghost_vendor_candidates = vendor_scorecard[
    vendor_scorecard["ghost_vendor_indicator"]
    | vendor_scorecard["risk_category"].eq("High")
].copy()
ghost_vendor_candidates.to_csv(OUTPUT_DIR / "ghost_vendor_candidates.csv", index=False)
ghost_vendor_candidates[[
    "vendor_id", "vendor_name", "vendor_risk_score", "risk_category", "days_since_registration",
    "total_invoice_value_jpy", "avg_invoice_z_score", "duplicate_bank_account", "is_ghost_vendor", "similar_to_vendor"
]].head(25)

In [ ]:
plt.figure(figsize=(11, 6))
sns.scatterplot(
    data=vendor_scorecard,
    x="invoice_count",
    y="total_invoice_value_jpy",
    hue="risk_category",
    size="vendor_risk_score",
    sizes=(40, 350),
    palette={"Low": "green", "Medium": "orange", "High": "red"},
)
plt.title("Vendor Risk Scorecard: Transaction Volume vs Value")
plt.xlabel("Invoice count")
plt.ylabel("Total invoice value (JPY)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=vendor_scorecard.head(10), y="vendor_name", x="vendor_risk_score", hue="risk_category", dodge=False)
plt.title("Top 10 Abnormal Vendor Risk Scores")
plt.xlabel("Risk score")
plt.ylabel("Vendor")
plt.tight_layout()
plt.show()